In [1]:
import pandas as pd

processed_path = "../data/processed"
df_roles = pd.read_csv(f"{processed_path}/role_skill_profiles.csv")

def compute_skill_gap(current_skills, target_onet_code):
    """
    Computes missing skills and match percentage between employee skills and target role requirements.
    """
    role_row = df_roles[df_roles['O*NET-SOC Code'] == target_onet_code]
    if role_row.empty:
        return {"error": "Target O*NET Code not found"}
    
    # Extract target requirements
    req_essential = role_row.iloc[0]['Required_Essential_Skills']
    req_software = role_row.iloc[0]['Required_Software_Skills']
    
    if isinstance(req_essential, str):
        req_essential = eval(req_essential) if req_essential else []
    if isinstance(req_software, str):
        req_software = eval(req_software) if req_software else []
        
    all_required = set([s.lower() for s in req_essential + req_software])
    current_set = set([s.lower() for s in current_skills])
    
    matched_skills = current_set.intersection(all_required)
    missing_skills = all_required - current_set
    
    match_score = (len(matched_skills) / len(all_required) * 100) if all_required else 100.0
    
    return {
        "target_role": role_row.iloc[0]['Title'],
        "match_score_pct": round(match_score, 2),
        "matched_count": len(matched_skills),
        "missing_count": len(missing_skills),
        "missing_skills_sample": list(missing_skills)[:10]
    }

# Test sample calculation on standard Software Developer role (e.g., 15-1252.00)
sample_employee_skills = ["Python", "SQL", "Critical Thinking", "Problem Solving"]
sample_target_code = df_roles['O*NET-SOC Code'].iloc[0]

result = compute_skill_gap(sample_employee_skills, sample_target_code)
print("=== SAMPLE SKILL GAP RESULT ===")
for k, v in result.items():
    print(f"{k}: {v}")

=== SAMPLE SKILL GAP RESULT ===
target_role: Chief Executives
match_score_pct: 1.75
matched_count: 1
missing_count: 56
missing_skills_sample: ['microsoft outlook', 'learning strategies', 'monitoring', 'microsoft excel', 'oracle peoplesoft', 'adobe acrobat', 'oracle e-business suite', 'sage 50 accounting', 'halogen e360', 'reading comprehension']
